In [218]:
# import
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
from scipy.stats import norm

In [219]:
# load data
df_raw = pd.read_csv("../data/test_data.csv")

# check
df_raw.head(10)

,Participant Code,Stable ID,Group Code,Task Group Code,Group Code Match,custom-fbclid,Submitted At,Task,Task Version,Task File Revision,...,sent,time_commit,time_end,time_render,time_run,time_show,time_switch,timestamp,type,url
0,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,13588.1,13587.0,8358.146,8372.4,8374.8,13642.1,2026-11-10T11:02:00.230Z,NaN,{}
1,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,14787.1,14783.0,13592.024,13588.5,13642.1,14841.7,2026-11-10T11:02:01.429Z,NaN,NaN
2,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,15160.6,15158.8,14775.190,14787.5,14841.7,15176.1,2026-11-10T11:02:01.803Z,NaN,NaN
3,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,15660.2,15658.8,15158.800,15160.8,15176.1,15674.8,2026-11-10T11:02:02.302Z,NaN,NaN
4,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,15836.2,15834.4,15658.800,15661.8,15674.8,15841.6,2026-11-10T11:02:02.478Z,NaN,NaN
5,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,16027.5,16025.4,15824.800,15837.5,15841.6,16041.6,2026-11-10T11:02:02.670Z,NaN,NaN
6,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,16027.9,16025.4,14775.190,14787.4,14841.7,16041.6,2026-11-10T11:02:02.670Z,NaN,NaN
7,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,16343.7,16342.2,16025.400,16028.1,16041.6,16358.1,2026-11-10T11:02:02.986Z,NaN,NaN
8,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,16493.6,16492.3,16342.200,16343.9,16358.1,16508.4,2026-11-10T11:02:03.136Z,NaN,NaN
9,58MSSZ47,AXMHJZ9,NaN,NaN,NaN,NaN,2026-11-10T11:03:14.112Z,Grammaticality Judgement,1.0,2,...,NaN,16509.3,16507.1,16492.300,16494.5,NaN,16524.9,2026-11-10T11:02:03.151Z,NaN,NaN


In [220]:
# define function for initial filtering
def filter_data(df):
    # rename relevant columns
    df = df.rename(columns = {"Participant Code": "subject",
                              "sender" : "screen_type",
                              "duration" : "screen_duration",
                              "sent" : "sequence",
                              "cond": "condition",
                              "response" : "response",
                              "correctResponse": "correct_response",
                              "correct": "accuracy"})
    # filter to relevant columns
    df = df[["subject", "screen_type", "screen_duration", "sequence", 
             "condition", "response", "correct_response", "accuracy"]]
    # filter out instr screens, practice rounds, etc by keeping only rows with a sequence
    df = df.dropna(subset = ["sequence"])
    return df



In [221]:
# apply filter function to raw data
df_filtered = filter_data(df_raw)

# check
df_filtered.head(10)

,subject,screen_type,screen_duration,sequence,condition,response,correct_response,accuracy
24,58MSSZ47,fx,383.3,Det lille kat var fint,u,NaN,NaN,NaN
25,58MSSZ47,stimuli,49.9,Det lille kat var fint,u,NaN,NaN,NaN
26,58MSSZ47,mask,204.9,Det lille kat var fint,u,u,u,True
27,58MSSZ47,fb,199.9,Det lille kat var fint,u,NaN,NaN,NaN
28,58MSSZ47,trial,834.0,Det lille kat var fint,u,NaN,NaN,NaN
29,58MSSZ47,fx,317.3,Min bil er ikke hvid,g,NaN,NaN,NaN
30,58MSSZ47,stimuli,226.7,Min bil er ikke hvid,g,g,g,True
31,58MSSZ47,mask,NaN,Min bil er ikke hvid,g,NaN,NaN,NaN
32,58MSSZ47,fb,200.1,Min bil er ikke hvid,g,NaN,NaN,NaN
33,58MSSZ47,trial,750.3,Min bil er ikke hvid,g,NaN,NaN,NaN


In [222]:
# define function for compressing data to one row per trial
def compress_data(trial):
    # get presentation duration as logged on stimuli screen
    pres_dur = trial.loc[trial["screen_type"] == "stimuli", 
                         "screen_duration"].values[0]
    # round presentation duration to nearest 50 (raw data is off due to screen refresh rate)
    pres_dur = round(pres_dur / 50) * 50
    # calculate RT as sum of raw presentation & mask duration
    rt = trial.loc[trial["screen_type"].isin(["stimuli", "mask"]), 
                   "screen_duration"].sum()
    # get accuracy, response, and correct response as logged on mask screen
    acc = trial.loc[trial["screen_type"] == "mask", 
                    "accuracy"].values[0]
    res = trial.loc[trial["screen_type"] == "mask", 
                    "response"].values[0]
    cor_res = trial.loc[trial["screen_type"] == "mask", 
                        "correct_response"].values[0]
    # get grammaticality condition (invariant for the trial)
    cond = trial["condition"].values[0]
    # return
    return pd.Series({"presentation_duration": pres_dur,
                      "RT": rt,
                      "condition": cond,
                      "response": res,
                      "correct_response" : cor_res,
                      "accuracy": acc})

In [223]:
# apply compression function at trial level as defined by subject & sequence
df_compressed = df_filtered.groupby(["subject", "sequence"]).apply(compress_data).reset_index()

# check
df_compressed.head(10)

,subject,sequence,presentation_duration,RT,condition,response,correct_response,accuracy
0,58MSSZ47,Bilen er stor og grim,100,113.3,g,NaN,NaN,NaN
1,58MSSZ47,Bilen ramte den lille pige,150,583.7,g,u,g,False
2,58MSSZ47,De driftig vand til maden,0,4.3,u,NaN,NaN,NaN
3,58MSSZ47,De drikker kaffe på altanen,150,216.9,g,u,g,False
4,58MSSZ47,De en fik ny hund,200,296.3,u,u,u,True
5,58MSSZ47,De gik en lang tur,250,272.5,g,u,g,False
6,58MSSZ47,De kan måske lide ham,200,187.8,g,NaN,NaN,NaN
7,58MSSZ47,De skal handle i Tyskland,150,136.3,g,NaN,NaN,NaN
8,58MSSZ47,De spisede mad i går,250,310.3,u,u,u,True
9,58MSSZ47,De vil ikke spiser mad,0,11.7,u,NaN,NaN,NaN


In [224]:
# define function for excluding participants and sequences with low mean accuracy
def filter_accuracy(df, threshold = 0.5):
    # participants
    mean_subject_acc = df.groupby("subject")["accuracy"].mean() 
    valid_subjects = mean_subject_acc[mean_subject_acc >= threshold].index
    df = df[df["subject"].isin(valid_subjects)]
    # sequences
    mean_sequence_acc = df.groupby("sequence")["accuracy"].mean()
    valid_sequences = mean_sequence_acc[mean_sequence_acc >= threshold].index
    df = df[df["sequence"].isin(valid_sequences)]
    return df

In [225]:
# apply accuracy filter function
df_clean = filter_accuracy(df_compressed, 0.0)

# check
df_clean.head(10)

,subject,sequence,presentation_duration,RT,condition,response,correct_response,accuracy
0,58MSSZ47,Bilen er stor og grim,100,113.3,g,NaN,NaN,NaN
1,58MSSZ47,Bilen ramte den lille pige,150,583.7,g,u,g,False
2,58MSSZ47,De driftig vand til maden,0,4.3,u,NaN,NaN,NaN
3,58MSSZ47,De drikker kaffe på altanen,150,216.9,g,u,g,False
4,58MSSZ47,De en fik ny hund,200,296.3,u,u,u,True
5,58MSSZ47,De gik en lang tur,250,272.5,g,u,g,False
6,58MSSZ47,De kan måske lide ham,200,187.8,g,NaN,NaN,NaN
7,58MSSZ47,De skal handle i Tyskland,150,136.3,g,NaN,NaN,NaN
8,58MSSZ47,De spisede mad i går,250,310.3,u,u,u,True
9,58MSSZ47,De vil ikke spiser mad,0,11.7,u,NaN,NaN,NaN


In [226]:
# define function for calculating average accuracy
def calculate_accuracy(group):
    acc = group["accuracy"].mean()
    return pd.Series({"accuracy": acc})

In [227]:
# apply function for calculating average accuracy at subject&presentation duration&condition level
df_acc = df_clean.groupby(["subject", "presentation_duration", "condition"]).apply(calculate_accuracy).reset_index()

In [228]:
# define function for determining response type outcome
def get_outcome(df):
    # set up confusion matrix for possible outcomes
    matrix = [(df["condition"] == "g") & (df["accuracy"] == True),   # hit
              (df["condition"] == "g") & (df["accuracy"] == False),  # miss
              (df["condition"] == "u") & (df["accuracy"] == True),   # correct rejection
              (df["condition"] == "u") & (df["accuracy"] == False)]  # false alarm
    # define outcome labels
    labels = ["hit", "miss", "CR", "FA"]
    # select label for each row and add to new outcome column
    df["outcome"] = np.select(matrix, labels, default = "unknown")
    return df

In [ ]:
# apply outcome determining function
df_outcomes = get_outcome(df_clean) 

# check
df_outcomes.head(10)

,subject,sequence,presentation_duration,RT,condition,response,correct_response,accuracy,outcome
0,58MSSZ47,Bilen er stor og grim,100,113.3,g,NaN,NaN,NaN,unknown
1,58MSSZ47,Bilen ramte den lille pige,150,583.7,g,u,g,False,miss
2,58MSSZ47,De driftig vand til maden,0,4.3,u,NaN,NaN,NaN,unknown
3,58MSSZ47,De drikker kaffe på altanen,150,216.9,g,u,g,False,miss
4,58MSSZ47,De en fik ny hund,200,296.3,u,u,u,True,CR
5,58MSSZ47,De gik en lang tur,250,272.5,g,u,g,False,miss
6,58MSSZ47,De kan måske lide ham,200,187.8,g,NaN,NaN,NaN,unknown
7,58MSSZ47,De skal handle i Tyskland,150,136.3,g,NaN,NaN,NaN,unknown
8,58MSSZ47,De spisede mad i går,250,310.3,u,u,u,True,CR
9,58MSSZ47,De vil ikke spiser mad,0,11.7,u,NaN,NaN,NaN,unknown


In [ ]:
# calc RTs

In [ ]:
# calc hit & FA rates

In [ ]:
# calc d' and c

In [ ]:
# plot accuracy

In [ ]:
# plot d' and c